# Translate Experiment Instructions

For the study instructions, use Facebook model to translate the experiment to start. 

In [1]:
import os
import torch
import pandas as pd

def translate_with_backtranslation(
    template_path,
    output_base_dir,
    target_lang="ukr",
    source_lang="eng",
    batch_size=8
):
    df = pd.read_excel(template_path)
    col = df.columns[0]

    texts = df[col].astype(str).str.strip().tolist()

    forward_translations = []
    back_translations = []

    # -------------------------
    # 1) Forward translation
    # -------------------------
    for i in tqdm(range(0, len(texts), batch_size), desc="Forward translation"):
        batch = texts[i:i+batch_size]

        inputs = processor(
            text=batch,
            src_lang=source_lang,
            return_tensors="pt",
            padding=True
        ).to(device)

        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                tgt_lang=target_lang,
                max_length=512
            )

        decoded = processor.batch_decode(
            output_tokens,
            skip_special_tokens=True
        )

        decoded = [t.strip() for t in decoded]
        forward_translations.extend(decoded)

    # -------------------------
    # 2) Back translation
    # -------------------------
    for i in tqdm(range(0, len(forward_translations), batch_size), desc="Back translation"):
        batch = forward_translations[i:i+batch_size]

        inputs = processor(
            text=batch,
            src_lang=target_lang,
            return_tensors="pt",
            padding=True
        ).to(device)

        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                tgt_lang=source_lang,
                max_length=512
            )

        decoded = processor.batch_decode(
            output_tokens,
            skip_special_tokens=True
        )

        decoded = [t.strip() for t in decoded]
        back_translations.extend(decoded)

    # -------------------------
    # 3) Save output
    # -------------------------
    out_df = pd.DataFrame({
        "English": texts,
        "translation": forward_translations,
        "back_translation": back_translations
    })

    output_dir = os.path.join(output_base_dir, target_lang)
    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(output_dir, f"{target_lang}_experiment.csv")
    out_df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

    return out_df

# Libraries and Models

In [2]:
import pandas as pd
import os
import torch
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForSeq2SeqLM


model_name = "facebook/seamless-m4t-v2-large"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

SeamlessM4Tv2ForTextToText(
  (shared): Embedding(256102, 1024, padding_idx=0)
  (text_encoder): SeamlessM4Tv2Encoder(
    (embed_tokens): SeamlessM4Tv2ScaledWordEmbedding(256102, 1024, padding_idx=0)
    (embed_positions): SeamlessM4Tv2SinusoidalPositionalEmbedding()
    (layers): ModuleList(
      (0-23): 24 x SeamlessM4Tv2EncoderLayer(
        (self_attn): SeamlessM4Tv2Attention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (ffn): SeamlessM4Tv2FeedForwardNetwork(
          (fc1): Linear(in_features=1024, out_features=8192, bias=True)
          (fc2): Linear(in_features=8192,

In [3]:
# translate into ukr
translate_with_backtranslation(
    template_path="task_translation_template.xlsx",
    output_base_dir="",
    target_lang="ukr"
)

Back translation: 100%|██████████| 13/13 [03:04<00:00, 14.22s/it]

Saved: ukr/ukr_experiment.csv


,English,translation,back_translation
0,Age,Вік,Age of the child
1,Age of Acquisition Study,Дослідження віку придбання,Age of acquisition research
2,Always think of how concrete (experience based...,"Завжди думайте про те, наскільки конкретним (н...",Always think about how specific (based on expe...
3,Are you primarily left- or right-handed?,Ви переважно ліворукий чи праворукий?,Are you predominantly left-handed or right-han...
4,Arousal ratings,Рейтинги збудження,The excitement ratings
...,...,...,...
92,You have completed the word ratings.,Ви завершили оцінку слів.,You've completed the word evaluation.
93,You may be compensated when taking part in thi...,Ви можете отримати компенсацію за участь у цьо...,You can get compensation for participating in ...
94,You will take this study entirely online from ...,Ви будете брати це дослідження повністю онлайн...,You will take the study entirely online from y...
95,Your decision whether to participate will not ...,Ваше рішення про участь не вплине на ваші пото...,Your decision to participate will not affect y...
